# Étape 3 — Feature Engineering
## MASI Hybrid Forecasting System

Leakage-free derivative features for the HMM (Étape 4) and CNN-LSTM (Étape 5).

**Anti-leakage:** L1 (TRAIN-only scaler stats) · L3 (`shift(>=1)` everywhere, empirically tested) · L4/L6 (inherited) · L8 (GARCH params from TRAIN only; per-window refit deferred to Étape 5).

Only `log_return` (MASI's own return) is contemporaneous. Every other feature uses strictly past data.

## 1. Run the feature-engineering pipeline

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('.'))

import pandas as pd
from IPython.display import Image, display

import importlib.util as _ilu, sys as _sys; _spec = _ilu.spec_from_file_location('e3', '../scripts/03_feature_engineering.py'); e3 = _ilu.module_from_spec(_spec); _sys.modules['e3'] = e3; _spec.loader.exec_module(e3)
e3.main()

## 2. Feature catalogue

In [ ]:
meta = json.load(open(os.path.join(e3.FEAT_DIR, 'feature_metadata.json')))
print(f"{meta['n_features']} engineered features, target = {meta['target']}")
for grp, cols in meta['feature_groups'].items():
    print(f"\n  {grp}:")
    for c in cols:
        print(f"    - {c}")
print('\nContemporaneous (settled at MASI close):', meta['contemporaneous_features'])
print('\nREJECTED features:')
for k, v in meta['rejected_features'].items():
    print(f'  - {k}: {v}')

## 3. Leakage / causality test (D5)

In [ ]:
pd.DataFrame(meta['leakage_test'])

## 4. RF sanity-check — engineered features vs Étape 2 raw features

In [ ]:
rf = json.load(open(os.path.join(e3.FEAT_DIR, 'rf_sanitycheck_metrics.json')))
ref = rf['etape2_rf_reference']
rows = []
for part in ('val', 'test'):
    m = rf[part]
    rows.append({'set': part.upper(), 'RMSE': round(m['rmse'], 6), 'MAE': round(m['mae'], 6),
                 'DA': round(m['directional_accuracy'], 4),
                 'DA_CI': [round(x, 4) for x in m['da_ci']],
                 'Sharpe': round(m['sharpe_annualized'], 3),
                 'MDD': round(m['max_drawdown'], 3), 'trades': m['n_trades']})
display(pd.DataFrame(rows).set_index('set'))
print(f"Étape 2 RF reference  — VAL DA={ref['val_da']:.4f}  TEST DA={ref['test_da']:.4f}  TEST Sharpe={ref['test_sharpe']:+.3f}")
print(f"Étape 3 RF (engineered) — TEST DA={rf['test']['directional_accuracy']:.4f}  delta={rf['test']['directional_accuracy']-ref['test_da']:+.4f}")

## 5. RF feature importance (recommended CNN-LSTM core-15)

In [ ]:
imp = rf['feature_importance']
imp_df = pd.DataFrame(sorted(imp.items(), key=lambda kv: kv[1], reverse=True),
                      columns=['feature', 'importance'])
imp_df['core_15'] = ['*' if i < 15 else '' for i in range(len(imp_df))]
display(imp_df)

## 6. Correlation heatmap (TRAIN)

In [ ]:
Image(os.path.join(e3.PLOTS_DIR, 'etape3_corr_heatmap.png'))

## 7. RF feature importance plot

In [ ]:
Image(os.path.join(e3.PLOTS_DIR, 'etape3_rf_importance.png'))

## 8. Volatility proxies (no VIX — constraint C1)

In [ ]:
Image(os.path.join(e3.PLOTS_DIR, 'etape3_volatility_proxies.png'))

## 9. Technical / momentum feature overview

In [ ]:
Image(os.path.join(e3.PLOTS_DIR, 'etape3_feature_overview.png'))

## 10. Takeaways for Étape 4 (HMM)

| Insight | Implication |
|---------|-------------|
| All causality cuts passed (max diff = 0.0) | Feature matrix is leakage-free — safe for HMM/CNN-LSTM |
| GARCH(1,1) α+β ≈ 0.90 | Strong volatility persistence — `garch_vol` is a meaningful regime signal |
| RF TEST DA improved +0.95 pp with engineered features | Feature engineering adds skill — proceed to HMM |
| Core-15 set identified by RF importance | Use as CNN-LSTM input (constraint C3, F ≤ 15) |
| GARCH params frozen from TRAIN | Étape 4/5 must re-fit GARCH per walk-forward window (L8) |